# Databricks Host and Token
### The Personal Access Token is used to authenticate your SDK calls. It's a secret key, so treat it like a password!
1. In your Databricks workspace, click your username in the top-right corner.
2. Select Settings.
3. Click the Developer tab.
4. Next to Access tokens, click Manage.
5. Click the Generate new token button.
6. Enter a Comment (e.g., "SDK Access") and set a Lifetime for the token (the recommended practice is to set an expiration date).
7. Click Generate.
8. Immediately copy the displayed token. This is the only time Databricks will show you the token. If you lose it, you'll have to generate a new one.
9. Add them to the databricksconfig file generated by the cli install.

In [ ]:
import os
from databricks.sdk import WorkspaceClient
import datetime as dt

# https://github.com/AgDMALabs-Public/ag-vision-dataops
from ag_vision.mobile.ingest import MobileImageIngest

# Connect to Roboflow

In [ ]:
w = WorkspaceClient(profile='agpile')  # set your profile
w.config.host

# Define Local and Databricks Variables.

In [ ]:
DATA_LOCAL_PATH = "/Users/danielwilliams/Documents/project data/board data/" # EX: '/Users/danielwilliams/Documents/project data/board data/'
DB_PROJECT_DIR = "/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data" # EX: '/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data'

COLLECTION_DATE = dt.datetime.now().strftime('%Y-%m-%d')
TRIAL = 'test'
SITE = 'California-labs'
FIELD = 'home'
LOCATION = 'home_1'

TASK = 'grain_imaging'
PROTOCOL = 'corn_blue_board'

YEAR = 2025
COUNTRY = 'IND'
CROP = 'maize'
TIME_OF_YEAR = 'spring'

# List out the files you want to upload

In [ ]:
files = os.listdir(DATA_LOCAL_PATH)
files = [x for x in files if '.jpeg' in x]
files = [DATA_LOCAL_PATH + x for x in files]
files

In [ ]:
# create an empty MobileImageIngest object.
ingest = MobileImageIngest(platform='local',
                           cloud_client=w,
                           cloud_bucket=None,
                           ingest_df=None)

In [ ]:
ingest.generate_ingest_df(file_list=files)


In [ ]:
ingest.ingest_df.head()


In [ ]:
# generate unique ID's for each of the images. This method used UUID4()
ingest.generate_unique_image_ids()

In [ ]:
# these are all the columns that you need to upload mobile scouting data to Fairgorunds and follow the Data Architecture Laid out in ag_vision.
ingest.ingest_df['project_dir'] = DB_PROJECT_DIR
ingest.ingest_df['site'] = SITE
ingest.ingest_df['trial'] = TRIAL
ingest.ingest_df['year'] = YEAR
ingest.ingest_df['country'] = COUNTRY
ingest.ingest_df['crop'] = CROP
ingest.ingest_df['time_of_year'] = TIME_OF_YEAR
ingest.ingest_df['field'] = FIELD
ingest.ingest_df['location'] = LOCATION
ingest.ingest_df['task'] = TASK
ingest.ingest_df['protocol'] = PROTOCOL
ingest.ingest_df['event_type'] = 'scouting'
ingest.ingest_df['collection_date'] = COLLECTION_DATE
ingest.ingest_df['plot_id'] = 'none' # if you want to upload event_type of trial you will need plot ID's for each image.

In [ ]:
ingest.ingest_df.head()


In [ ]:
# From the year, country, crop, and time of year we can generate a season code.
ingest.add_season_column()

In [ ]:
# Validate that all the columns are there and for some columns valdiate what is in them.
ingest.validate_ingest_df()

In [ ]:
# generate the dst_path from all the data in the column of ingest_df.
ingest.generate_dst_path_name()

In [ ]:
ingest.ingest_df.head()


In [ ]:
ingest.ingest_df['dst_path'][0]

In [ ]:
# this uploads the data to the cloud. If files failed they will be stored in the failed_upload list.
ingest.upload_local_data_to_db()

In [ ]:
print(ingest.cloud_client)